In [1]:
import pandas as pd
import numpy as np
from dep2pyodbc import dep2connection

pd.set_option("display.max_columns", None)

channel_crh = dep2connection("CRH")
channel_dwh_lisa = dep2connection("CRH_DWH")
cursor = channel_dwh_lisa.cursor()

pyodbc using windows
pyodbc using windows


In [2]:
mdq = pd.read_csv('./csv/raw_data/decoded_mdq.csv')
mdq.head()

,Unnamed: 0,CandidateID,InstanceID,itemId,first_val,second_val,third_val,fourth_val,fifth_val,sixth_val
0,0,108773,1,2,4,0,0,6,0,0
1,1,108773,1,3,4,0,0,6,0,0
2,2,108773,1,3,0,1,0,6,0,0
3,3,108773,1,1,0,1,0,6,0,0
4,4,108773,1,1,4,1,0,2,0,0


In [3]:
mdq.columns

Index(['Unnamed: 0', 'CandidateID', 'InstanceID', 'itemId', 'first_val',
       'second_val', 'third_val', 'fourth_val', 'fifth_val', 'sixth_val'],
      dtype='object')

In [4]:
df_mdq = mdq[['CandidateID', 'InstanceID', 'itemId', 'first_val',
       'second_val', 'third_val', 'fourth_val', 'fifth_val', 'sixth_val']]
df_mdq.rename(columns={
    'first_val': 'FirstVal',
    'second_val': 'SecondVal',
    'third_val': 'ThirdVal',
    'fourth_val': 'FourthVal',
    'fifth_val': 'FifthVal',
    'sixth_val': 'SixthVal',
    }, inplace=True)
df_mdq['Test'] = 'MDQ'
df_mdq.head()

C:\Users\Mikel\AppData\Local\Temp\ipykernel_10468\2548690819.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_mdq.rename(columns={


,CandidateID,InstanceID,itemId,FirstVal,SecondVal,ThirdVal,FourthVal,FifthVal,SixthVal,Test
0,108773,1,2,4,0,0,6,0,0,MDQ
1,108773,1,3,4,0,0,6,0,0,MDQ
2,108773,1,3,0,1,0,6,0,0,MDQ
3,108773,1,1,0,1,0,6,0,0,MDQ
4,108773,1,1,4,1,0,2,0,0,MDQ


In [5]:
df_mdq.info()
df_mdq.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7727760 entries, 0 to 7727759
Data columns (total 10 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   CandidateID  int64 
 1   InstanceID   int64 
 2   itemId       int64 
 3   FirstVal     int64 
 4   SecondVal    int64 
 5   ThirdVal     int64 
 6   FourthVal    int64 
 7   FifthVal     int64 
 8   SixthVal     int64 
 9   Test         object
dtypes: int64(9), object(1)
memory usage: 589.6+ MB


Index(['CandidateID', 'InstanceID', 'itemId', 'FirstVal', 'SecondVal',
       'ThirdVal', 'FourthVal', 'FifthVal', 'SixthVal', 'Test'],
      dtype='object')

## Basic cleaning

In [6]:
df_mdq['answer_sequence'] = df_mdq[['FirstVal', 'SecondVal', 'ThirdVal', 'FourthVal', 'FifthVal', 'SixthVal']].values.tolist()

In [7]:
df_mdq.info()
df_mdq.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7727760 entries, 0 to 7727759
Data columns (total 11 columns):
 #   Column           Dtype 
---  ------           ----- 
 0   CandidateID      int64 
 1   InstanceID       int64 
 2   itemId           int64 
 3   FirstVal         int64 
 4   SecondVal        int64 
 5   ThirdVal         int64 
 6   FourthVal        int64 
 7   FifthVal         int64 
 8   SixthVal         int64 
 9   Test             object
 10  answer_sequence  object
dtypes: int64(9), object(2)
memory usage: 648.5+ MB


,CandidateID,InstanceID,itemId,FirstVal,SecondVal,ThirdVal,FourthVal,FifthVal,SixthVal,Test,answer_sequence
0,108773,1,2,4,0,0,6,0,0,MDQ,"[4, 0, 0, 6, 0, 0]"
1,108773,1,3,4,0,0,6,0,0,MDQ,"[4, 0, 0, 6, 0, 0]"
2,108773,1,3,0,1,0,6,0,0,MDQ,"[0, 1, 0, 6, 0, 0]"
3,108773,1,1,0,1,0,6,0,0,MDQ,"[0, 1, 0, 6, 0, 0]"
4,108773,1,1,4,1,0,2,0,0,MDQ,"[4, 1, 0, 2, 0, 0]"


In [8]:
df_mdq = df_mdq[df_mdq['answer_sequence'].apply(lambda x: any(x))]
df_mdq.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3348455 entries, 0 to 7727742
Data columns (total 11 columns):
 #   Column           Dtype 
---  ------           ----- 
 0   CandidateID      int64 
 1   InstanceID       int64 
 2   itemId           int64 
 3   FirstVal         int64 
 4   SecondVal        int64 
 5   ThirdVal         int64 
 6   FourthVal        int64 
 7   FifthVal         int64 
 8   SixthVal         int64 
 9   Test             object
 10  answer_sequence  object
dtypes: int64(9), object(2)
memory usage: 306.6+ MB


In [9]:
df_mdq.drop(columns=['answer_sequence'], inplace=True)

## Adding keys

### Candidates 

In [10]:
df_candidates_before_key = pd.read_sql("SELECT ID, CandidateID, InstanceID, CreatedDate, ModifiedDate, VersionNumber FROM CandidateResultMotivation", channel_crh)
df_candidates_before_key.head()

C:\Users\Mikel\AppData\Local\Temp\ipykernel_10468\942627830.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_candidates_before_key = pd.read_sql("SELECT ID, CandidateID, InstanceID, CreatedDate, ModifiedDate, VersionNumber FROM CandidateResultMotivation", channel_crh)


,ID,CandidateID,InstanceID,CreatedDate,ModifiedDate,VersionNumber
0,1,108773,1,2007-12-28 14:22:20.837,2007-12-28 14:30:10.013,None
1,2,126554,1,2007-12-28 14:26:26.643,2007-12-28 14:36:29.633,None
2,3,126720,1,2008-01-02 17:21:53.890,2008-01-02 17:36:13.653,None
3,4,127020,1,2008-01-04 13:53:21.310,2008-01-04 14:06:43.507,None
4,5,127924,1,2008-01-11 13:35:07.883,2008-01-11 13:50:33.123,None


In [11]:
df_mdq = pd.merge(df_mdq, df_candidates_before_key, on=["CandidateID", "InstanceID"], how="left")
df_mdq.head()

,CandidateID,InstanceID,itemId,FirstVal,SecondVal,ThirdVal,FourthVal,FifthVal,SixthVal,Test,ID,CreatedDate,ModifiedDate,VersionNumber
0,108773,1,2,4,0,0,6,0,0,MDQ,1,2007-12-28 14:22:20.837,2007-12-28 14:30:10.013,None
1,108773,1,3,4,0,0,6,0,0,MDQ,1,2007-12-28 14:22:20.837,2007-12-28 14:30:10.013,None
2,108773,1,3,0,1,0,6,0,0,MDQ,1,2007-12-28 14:22:20.837,2007-12-28 14:30:10.013,None
3,108773,1,1,0,1,0,6,0,0,MDQ,1,2007-12-28 14:22:20.837,2007-12-28 14:30:10.013,None
4,108773,1,1,4,1,0,2,0,0,MDQ,1,2007-12-28 14:22:20.837,2007-12-28 14:30:10.013,None


In [12]:
dim_candidate = pd.read_sql("SELECT CandidateKey, ID, InstanceID FROM DimCandidate", channel_dwh_lisa)
dim_candidate.head()

C:\Users\Mikel\AppData\Local\Temp\ipykernel_10468\977585254.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_candidate = pd.read_sql("SELECT CandidateKey, ID, InstanceID FROM DimCandidate", channel_dwh_lisa)


,CandidateKey,ID,InstanceID
0,14,1,4
1,22,2,2
2,23,2,3
3,24,2,4
4,32,3,2


In [13]:
df_mdq = pd.merge(df_mdq, dim_candidate, left_on=["CandidateID", "InstanceID"], right_on=["ID", "InstanceID"], how="left")
df_mdq.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3348611 entries, 0 to 3348610
Data columns (total 16 columns):
 #   Column         Dtype         
---  ------         -----         
 0   CandidateID    int64         
 1   InstanceID     int64         
 2   itemId         int64         
 3   FirstVal       int64         
 4   SecondVal      int64         
 5   ThirdVal       int64         
 6   FourthVal      int64         
 7   FifthVal       int64         
 8   SixthVal       int64         
 9   Test           object        
 10  ID_x           int64         
 11  CreatedDate    datetime64[ns]
 12  ModifiedDate   datetime64[ns]
 13  VersionNumber  object        
 14  CandidateKey   int64         
 15  ID_y           int64         
dtypes: datetime64[ns](2), int64(12), object(2)
memory usage: 408.8+ MB


In [14]:
df_mdq.drop(columns=["ID_y", "CandidateID"], inplace=True)
df_mdq.rename(columns={"ID_x": "TestID"}, inplace=True)
df_mdq.head()

,InstanceID,itemId,FirstVal,SecondVal,ThirdVal,FourthVal,FifthVal,SixthVal,Test,TestID,CreatedDate,ModifiedDate,VersionNumber,CandidateKey
0,1,2,4,0,0,6,0,0,MDQ,1,2007-12-28 14:22:20.837,2007-12-28 14:30:10.013,None,1087731
1,1,3,4,0,0,6,0,0,MDQ,1,2007-12-28 14:22:20.837,2007-12-28 14:30:10.013,None,1087731
2,1,3,0,1,0,6,0,0,MDQ,1,2007-12-28 14:22:20.837,2007-12-28 14:30:10.013,None,1087731
3,1,1,0,1,0,6,0,0,MDQ,1,2007-12-28 14:22:20.837,2007-12-28 14:30:10.013,None,1087731
4,1,1,4,1,0,2,0,0,MDQ,1,2007-12-28 14:22:20.837,2007-12-28 14:30:10.013,None,1087731


### Dates

In [15]:
df_dates = pd.read_sql("SELECT DateKey, Date FROM DimDate", channel_dwh_lisa)
df_dates.head()

C:\Users\Mikel\AppData\Local\Temp\ipykernel_10468\594440161.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dates = pd.read_sql("SELECT DateKey, Date FROM DimDate", channel_dwh_lisa)


,DateKey,Date
0,20110101,2011-01-01
1,20110102,2011-01-02
2,20110103,2011-01-03
3,20110104,2011-01-04
4,20110105,2011-01-05


In [16]:
df_mdq["CreatedDate"] = pd.to_datetime(df_mdq["CreatedDate"]).dt.date

In [17]:
df_mdq.head()

,InstanceID,itemId,FirstVal,SecondVal,ThirdVal,FourthVal,FifthVal,SixthVal,Test,TestID,CreatedDate,ModifiedDate,VersionNumber,CandidateKey
0,1,2,4,0,0,6,0,0,MDQ,1,2007-12-28,2007-12-28 14:30:10.013,None,1087731
1,1,3,4,0,0,6,0,0,MDQ,1,2007-12-28,2007-12-28 14:30:10.013,None,1087731
2,1,3,0,1,0,6,0,0,MDQ,1,2007-12-28,2007-12-28 14:30:10.013,None,1087731
3,1,1,0,1,0,6,0,0,MDQ,1,2007-12-28,2007-12-28 14:30:10.013,None,1087731
4,1,1,4,1,0,2,0,0,MDQ,1,2007-12-28,2007-12-28 14:30:10.013,None,1087731


In [18]:
# df_mdq.sort_values(by='CreatedDate').head()

In [19]:
df_mdq = pd.merge(df_mdq, df_dates, left_on="CreatedDate", right_on="Date", how="left")
df_mdq.drop(columns=["Date", "CreatedDate"], inplace=True)
df_mdq.rename(columns={"DateKey": "CreatedDateKey"}, inplace=True)

In [20]:
df_mdq["ModifiedDate"] = pd.to_datetime(df_mdq["ModifiedDate"]).dt.date
df_mdq = pd.merge(df_mdq, df_dates, left_on="ModifiedDate", right_on="Date", how="left")
df_mdq.drop(columns=["Date", "ModifiedDate"], inplace=True)
df_mdq.rename(columns={"DateKey": "ModifiedDateKey"}, inplace=True)

In [21]:
df_mdq.head()

,InstanceID,itemId,FirstVal,SecondVal,ThirdVal,FourthVal,FifthVal,SixthVal,Test,TestID,VersionNumber,CandidateKey,CreatedDateKey,ModifiedDateKey
0,1,2,4,0,0,6,0,0,MDQ,1,None,1087731,NaN,NaN
1,1,3,4,0,0,6,0,0,MDQ,1,None,1087731,NaN,NaN
2,1,3,0,1,0,6,0,0,MDQ,1,None,1087731,NaN,NaN
3,1,1,0,1,0,6,0,0,MDQ,1,None,1087731,NaN,NaN
4,1,1,4,1,0,2,0,0,MDQ,1,None,1087731,NaN,NaN


### Next TestKey available

In [23]:
df_test_baq = pd.read_csv('../decoded_data/BA51/FactTest.csv')
df_test_baq.head()

,Test,CandidateKey,CreatedDateKey,ModifiedDateKey,VersionNumber,TestKey
0,BAQ,1034431,NaN,NaN,NaN,183642
1,BAQ,1034091,NaN,NaN,NaN,183643
2,BAQ,1033721,NaN,NaN,NaN,183644
3,BAQ,1025991,NaN,NaN,NaN,183645
4,BAQ,1034191,NaN,NaN,NaN,183646


In [24]:
max_key = df_test_baq.TestKey.max()
max_key

255110

## FactTest

In [25]:
df_test = df_mdq[["TestID", "Test", "CandidateKey", "CreatedDateKey", "ModifiedDateKey", "VersionNumber"]]

df_test.drop_duplicates(inplace=True)
df_test.reset_index(inplace=True, drop=True)
df_test["TestKey"] = df_test.index + max_key + 1

df_test.head()

C:\Users\Mikel\AppData\Local\Temp\ipykernel_10468\1180813710.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.drop_duplicates(inplace=True)
C:\Users\Mikel\AppData\Local\Temp\ipykernel_10468\1180813710.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["TestKey"] = df_test.index + max_key + 1


,TestID,Test,CandidateKey,CreatedDateKey,ModifiedDateKey,VersionNumber,TestKey
0,1,MDQ,1087731,NaN,NaN,None,255111
1,2,MDQ,1265541,NaN,NaN,None,255112
2,3,MDQ,1267201,NaN,NaN,None,255113
3,4,MDQ,1270201,NaN,NaN,None,255114
4,5,MDQ,1279241,NaN,NaN,None,255115


## FactQuestionBAQ

In [26]:
df_question = df_mdq.drop(columns=["CandidateKey", "CreatedDateKey", "ModifiedDateKey", "VersionNumber"])
df_question["QuestionKey"] = df_question.index + 1
df_question.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3348611 entries, 0 to 3348610
Data columns (total 11 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   InstanceID   int64 
 1   itemId       int64 
 2   FirstVal     int64 
 3   SecondVal    int64 
 4   ThirdVal     int64 
 5   FourthVal    int64 
 6   FifthVal     int64 
 7   SixthVal     int64 
 8   Test         object
 9   TestID       int64 
 10  QuestionKey  int64 
dtypes: int64(10), object(1)
memory usage: 281.0+ MB


In [27]:
df_question.head()

,InstanceID,itemId,FirstVal,SecondVal,ThirdVal,FourthVal,FifthVal,SixthVal,Test,TestID,QuestionKey
0,1,2,4,0,0,6,0,0,MDQ,1,1
1,1,3,4,0,0,6,0,0,MDQ,1,2
2,1,3,0,1,0,6,0,0,MDQ,1,3
3,1,1,0,1,0,6,0,0,MDQ,1,4
4,1,1,4,1,0,2,0,0,MDQ,1,5


In [28]:
len(df_question)

3348611

In [29]:
df_question = pd.merge(df_question, df_test[["TestKey", "TestID"]], on="TestID", how="left")

In [30]:
df_question.head()

,InstanceID,itemId,FirstVal,SecondVal,ThirdVal,FourthVal,FifthVal,SixthVal,Test,TestID,QuestionKey,TestKey
0,1,2,4,0,0,6,0,0,MDQ,1,1,255111
1,1,3,4,0,0,6,0,0,MDQ,1,2,255111
2,1,3,0,1,0,6,0,0,MDQ,1,3,255111
3,1,1,0,1,0,6,0,0,MDQ,1,4,255111
4,1,1,4,1,0,2,0,0,MDQ,1,5,255111


In [31]:
df_question.drop(columns=['TestID', 'InstanceID'], inplace=True)

In [32]:
df_question.head()

,itemId,FirstVal,SecondVal,ThirdVal,FourthVal,FifthVal,SixthVal,Test,QuestionKey,TestKey
0,2,4,0,0,6,0,0,MDQ,1,255111
1,3,4,0,0,6,0,0,MDQ,2,255111
2,3,0,1,0,6,0,0,MDQ,3,255111
3,1,0,1,0,6,0,0,MDQ,4,255111
4,1,4,1,0,2,0,0,MDQ,5,255111


In [33]:
df_test.drop(columns=["TestID"], inplace=True)

C:\Users\Mikel\AppData\Local\Temp\ipykernel_10468\1141659774.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.drop(columns=["TestID"], inplace=True)


## To csv 

In [34]:
df_test.to_csv('../decoded_data/MDQ/FactTest.csv', index=False)
df_question.to_csv('../decoded_data/MDQ/FactQuestionMDQ.csv', index=False)